# Limpieza reproducible del conjunto de establecimientos

## Resumen

Este notebook aplica las reglas aprobadas en `03_plan_limpieza.md`. Produce un conjunto candidato para validación, el registro completo de transformaciones y la revisión caso por caso de posibles duplicados parciales. Ningún candidato parcial se elimina automáticamente.

## Configuración

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data" / "raw" / "establecimientos_raw.csv").exists():
    raise FileNotFoundError("Ejecute el notebook desde la raíz del repositorio o desde notebooks/.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

from src import catalogos, diagnostico, scraping, validadores
from src.duplicados import encontrar_duplicados_parciales
from src.pipeline_limpieza import limpiar_establecimientos

## Carga del conjunto crudo

In [2]:
RUTA_RAW = ROOT / "data" / "raw" / "establecimientos_raw.csv"
RUTA_CANDIDATO = ROOT / "data" / "processed" / "establecimientos_limpios_candidato.csv"
RUTA_TRANSFORMACIONES = ROOT / "data" / "processed" / "transformaciones.csv"
RUTA_DUPLICADOS = ROOT / "data" / "processed" / "duplicados_revisados.csv"

df_raw = pd.read_csv(RUTA_RAW, dtype="string", keep_default_na=False)
print("Dimensión cruda:", df_raw.shape)
diagnostico.resumen_faltantes_total(df_raw)

Dimensión cruda: (11891, 17)


celdas_faltantes             4636.00
celdas_totales             202147.00
porcentaje                      2.29
variables_con_faltantes        17.00
dtype: float64

## Línea base de posibles duplicados

In [3]:
filas_vacias_raw = df_raw.apply(diagnostico.mascara_faltantes).all(axis=1)
df_raw_real = df_raw.loc[~filas_vacias_raw].reset_index(drop=True)
duplicados_parciales_antes = encontrar_duplicados_parciales(df_raw_real)
registros_candidatos_antes = set(duplicados_parciales_antes["indice_1"]) | set(
    duplicados_parciales_antes["indice_2"]
)
print("Pares candidatos Antes:", len(duplicados_parciales_antes))
print("Registros involucrados Antes:", len(registros_candidatos_antes))

Pares candidatos Antes: 769
Registros involucrados Antes: 1372


## Aplicación determinista de las reglas

In [4]:
resultado = limpiar_establecimientos(df_raw)
df_limpio = resultado.datos
transformaciones = resultado.transformaciones
duplicados_revisados = resultado.duplicados_revisados

print("Dimensión candidata:", df_limpio.shape)
print("Transformaciones documentadas:", len(transformaciones))
print("Pares parciales revisados:", len(duplicados_revisados))

Dimensión candidata: (11868, 18)
Transformaciones documentadas: 28
Pares parciales revisados: 781


## Registro de transformaciones

In [5]:
transformaciones

,Variable,Problema detectado,Transformación,Registros afectados,Justificación
0,CODIGO,"Vacíos, marcadores de ausencia o formato textu...","Convertir marcadores a NA; normalizar Unicode,...",23,Distingue ausencia real de texto y deja una re...
1,DISTRITO,"Vacíos, marcadores de ausencia o formato textu...","Convertir marcadores a NA; normalizar Unicode,...",556,Distingue ausencia real de texto y deja una re...
2,DEPARTAMENTO,"Vacíos, marcadores de ausencia o formato textu...","Convertir marcadores a NA; normalizar Unicode,...",23,Distingue ausencia real de texto y deja una re...
3,MUNICIPIO,"Vacíos, marcadores de ausencia o formato textu...","Convertir marcadores a NA; normalizar Unicode,...",23,Distingue ausencia real de texto y deja una re...
4,ESTABLECIMIENTO,"Vacíos, marcadores de ausencia o formato textu...","Convertir marcadores a NA; normalizar Unicode,...",68,Distingue ausencia real de texto y deja una re...
5,DIRECCION,"Vacíos, marcadores de ausencia o formato textu...","Convertir marcadores a NA; normalizar Unicode,...",150,Distingue ausencia real de texto y deja una re...
6,TELEFONO,"Vacíos, marcadores de ausencia o formato textu...","Convertir marcadores a NA; normalizar Unicode,...",970,Distingue ausencia real de texto y deja una re...
7,SUPERVISOR,"Vacíos, marcadores de ausencia o formato textu...","Convertir marcadores a NA; normalizar Unicode,...",564,Distingue ausencia real de texto y deja una re...
8,DIRECTOR,"Vacíos, marcadores de ausencia o formato textu...","Convertir marcadores a NA; normalizar Unicode,...",2161,Distingue ausencia real de texto y deja una re...
9,NIVEL,"Vacíos, marcadores de ausencia o formato textu...","Convertir marcadores a NA; normalizar Unicode,...",23,Distingue ausencia real de texto y deja una re...


## Revisión conservadora de duplicados parciales

In [6]:
print(duplicados_revisados["decision"].value_counts(dropna=False))
registros_candidatos_despues = set(duplicados_revisados["indice_1"]) | set(
    duplicados_revisados["indice_2"]
)
print("Registros involucrados Después:", len(registros_candidatos_despues))
duplicados_revisados.head(20)

decision
CONSERVAR    781
Name: count, dtype: int64
Registros involucrados Después: 1386


,indice_1,indice_2,codigo_1,codigo_2,establecimiento_1,establecimiento_2,direccion_1,direccion_2,telefono_1,telefono_2,similitud_nombre,similitud_direccion,telefono_igual,campos_diferentes,decision,justificacion
0,20,21,15-01-0157-46,15-01-0160-46,LICEO SAN RAFAEL,LICEO SAN RAFAEL,ALDEA CHILASCÓ,ALDEA CHILASCÓ,50022615,50022615,100.0,100.00,True,NINGUNO,CONSERVAR,Los códigos MINEDUC son distintos y funcionan ...
1,74,75,15-03-0060-46,15-03-0062-46,COLEGIO PARTICULAR MIXTO SANTO DOMINGO,COLEGIO PARTICULAR MIXTO SANTO DOMINGO,4A. AVENIDA ENTRE 3A. Y 4A. CALLE ZONA 2,4A. AVENIDA ENTRE 3A. Y 4A. CALLE ZONA 2,79388037,79388037,100.0,100.00,True,"DISTRITO, SUPERVISOR, DIRECTOR",CONSERVAR,Los códigos MINEDUC son distintos y funcionan ...
2,82,98,15-03-0088-46,15-03-2053-46,COLEGIO PARTICULAR MIXTO VIDA NUEVA,COLEGIO PARTICULAR MIXTO VIDA NUEVA,4A. AVENIDA 1-83 ZONA 2,4A. AVENIDA 1-83 ZONA 2,58408911,NaN,100.0,100.00,False,TELEFONO,CONSERVAR,Los códigos MINEDUC son distintos y funcionan ...
3,83,100,15-03-0089-46,15-03-2167-46,CENTRO ESTUDIANTIL CON ORIENTACIÓN EN COMPUTACIÓN,CENTRO EDUCATIVO ESTUDIANTIL CON ORIENTACION E...,3RA. CALLE 5-41 ZONA 2,3A. CALLE 5-41 ZONA 2,79388223,79388223,100.0,97.56,True,"ESTABLECIMIENTO, DIRECCION",CONSERVAR,Los códigos MINEDUC son distintos y funcionan ...
4,90,93,15-03-0778-46,15-03-1888-46,CENTRO CULTURAL DE AMERICA,CENTRO CULTURAL DE AMERICA,3A. CALLE 3-64 ZONA 2,3A. CALLE 3-64 ZONA 2,79388718,79388718,100.0,100.00,True,NINGUNO,CONSERVAR,Los códigos MINEDUC son distintos y funcionan ...
5,115,116,15-04-0268-46,15-04-0269-46,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,ALDEA LOS PAJALES,"CASERÍO CHIRRAMOS, ALDEA LOS PAJALES",32276501,49565089,100.0,100.00,False,"DIRECCION, TELEFONO",CONSERVAR,Los códigos MINEDUC son distintos y funcionan ...
6,115,119,15-04-0268-46,15-04-0272-46,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,ALDEA LOS PAJALES,"CASERÍO CHITOMAX, ALDEA LOS PAJALES",32276501,53073375,100.0,100.00,False,"DISTRITO, DIRECCION, TELEFONO, SUPERVISOR",CONSERVAR,Los códigos MINEDUC son distintos y funcionan ...
7,116,119,15-04-0269-46,15-04-0272-46,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,"CASERÍO CHIRRAMOS, ALDEA LOS PAJALES","CASERÍO CHITOMAX, ALDEA LOS PAJALES",49565089,53073375,100.0,86.96,False,"DISTRITO, DIRECCION, TELEFONO, SUPERVISOR",CONSERVAR,Los códigos MINEDUC son distintos y funcionan ...
8,122,125,15-04-0275-46,15-04-0278-46,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,INED INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFI...,"CASERÍO LA ESTANCIA, ALDEA PATZIJÓN","CASERÍO LA LAGUNA PATZIJÓN, ALDEA PATZIJÓN",51615845,40864512,100.0,87.88,False,"DISTRITO, DIRECCION, TELEFONO, SUPERVISOR",CONSERVAR,Los códigos MINEDUC son distintos y funcionan ...
9,185,212,16-01-0481-46,16-01-0664-46,LICEO AMERICANO DEL NORTE,LICEO AMERICANO DEL NORTE,5TA. CALLE 2-23 ZONA 4,5TA. AVENIDA 2-23 ZONA 4,79514754,79514754,100.0,83.33,True,DIRECCION,CONSERVAR,Los códigos MINEDUC son distintos y funcionan ...


Los pares se conservan porque poseen códigos MINEDUC diferentes. El archivo registra similitudes, campos diferentes, decisión y justificación para cada caso; una fusión futura requeriría confirmación de la fuente.

## Comprobaciones de calidad del candidato

In [7]:
errores = validadores.contar_errores_calidad(df_limpio)
errores

{'duplicados_exactos': 0,
 'espacios_extra': 0,
 'telefonos_invalidos': 0,
 'codigos_invalidos': 0,
 'distritos_invalidos': 0,
 'departamentos_invalidos': 0,
 'municipios_invalidos': 0,
 'relaciones_departamentales_invalidas': 0,
 'columnas_requeridas_ausentes': 0}

In [8]:
assert all(valor == 0 for valor in errores.values()), errores
assert df_limpio["CODIGO"].is_unique
assert df_limpio["NIVEL"].eq("DIVERSIFICADO").all()
assert set(df_limpio["DEPARTAMENTO"].dropna()) == set(catalogos.DEPARTAMENTOS_OFICIALES)
print("Comprobaciones internas aprobadas.")

Comprobaciones internas aprobadas.


## Comparación preliminar de calidad

In [9]:
comparacion_faltantes = pd.DataFrame(
    {
        "Antes": diagnostico.resumen_faltantes_total(df_raw),
        "Después_candidato": diagnostico.resumen_faltantes_total(df_limpio),
    }
)
comparacion_faltantes

,Antes,Después_candidato
celdas_faltantes,4636.00,14139.00
celdas_totales,202147.00,213624.00
porcentaje,2.29,6.62
variables_con_faltantes,17.00,7.00


## Exportación de artefactos

In [10]:
RUTA_CANDIDATO.parent.mkdir(parents=True, exist_ok=True)
RUTA_TRANSFORMACIONES.parent.mkdir(parents=True, exist_ok=True)

df_limpio.to_csv(RUTA_CANDIDATO, index=False)
transformaciones.to_csv(RUTA_TRANSFORMACIONES, index=False)
duplicados_revisados.to_csv(RUTA_DUPLICADOS, index=False)

print("Candidato:", RUTA_CANDIDATO.relative_to(ROOT))
print("SHA-256 candidato:", scraping.sha256_archivo(RUTA_CANDIDATO))
print("Transformaciones:", RUTA_TRANSFORMACIONES.relative_to(ROOT))
print("Duplicados revisados:", RUTA_DUPLICADOS.relative_to(ROOT))

Candidato: data/processed/establecimientos_limpios_candidato.csv
SHA-256 candidato: 0364f39ec1f9d07e4fb36bba5474ba8b2973fb69a2dbfb9c29ae5bc14d7a3a43
Transformaciones: data/processed/transformaciones.csv
Duplicados revisados: data/processed/duplicados_revisados.csv


## Resultado

El conjunto candidato conserva una fila por código oficial, añade `ZONA_CAPITAL`, normaliza geografía y formatos, representa las ausencias con `NA` y supera las comprobaciones internas. La validación independiente y la exportación del CSV final pertenecen a las fases siguientes.